# A widget from an op

- The op declares `sigma`, with its range.
- opspec reads that off the signature.
- The notebook builds a slider from it.
- No GUI code per op. No napari, no Qt.

Needs `pip install -e .`, plus numpy, scikit-image, matplotlib and ipywidgets.

In [ ]:
from typing import Annotated

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from skimage import data, filters, measure
from skimage.color import label2rgb

from opspec.op import OpSpec, Role, op

Image = Annotated[np.ndarray, Role.image]
Labels = Annotated[np.ndarray, Role.labels]
Sigma = Annotated[float, {"min": 0.1, "max": 10.0, "step": 0.1}]
coins = data.coins()

## The ops

Same two as notebook 1.

In [ ]:
@op
def smooth(image: Image, sigma: Sigma = 2.0) -> Image:
    """Blur an image."""
    return filters.gaussian(image, sigma=sigma)


@op
def label_objects(image: Image, sigma: Sigma = 2.0) -> Labels:
    """Blur, threshold, then number each connected object."""
    blurred = filters.gaussian(image, sigma=sigma)
    return measure.label(blurred > filters.threshold_otsu(blurred))

## Build the widget

- One slider per `float` parameter.
- Draw by role: `image` is grey, `labels` is random colours.
- Both read from the spec, so neither mentions a specific op.

In [ ]:
def draw(the_op, result):
    role = OpSpec.from_op(the_op).return_role
    plt.figure(figsize=(5, 4))
    if role is Role.labels:
        plt.imshow(label2rgb(result, bg_label=0))
    else:
        plt.imshow(result, cmap="gray")
    plt.axis("off")
    plt.show()


def auto_widget(the_op, image):
    spec = OpSpec.from_op(the_op)
    sliders = {
        p.name: widgets.FloatSlider(value=p.default, **p.ui)
        for p in spec.params
        if p.type is float
    }
    return widgets.interactive(
        lambda **kwargs: draw(the_op, the_op(image, **kwargs)), **sliders
    )

In [ ]:
auto_widget(smooth, coins)

In [ ]:
auto_widget(label_objects, coins)

## What this shows

- Same `auto_widget` for both ops. It never names one.
- The second op draws in colour because it declared `Role.labels`.
- Add a parameter to an op, get a slider. Delete one, lose the slider.
- The slider range comes from the op, not from this notebook.